In [1]:
# %pip install faiss-cpu llama-index-vector-stores-faiss llama-index-llms-ollama llama-index-embeddings-ollama

In [2]:
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext, load_index_from_storage

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [3]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model = "gemma2:2b",
    temperature = 0.5,
    request_timeout = 120.0
)

embed_model = OllamaEmbedding(
    model_name = "nomic-embed-text",
)

In [4]:
# 데이터 로드
documents = SimpleDirectoryReader("../Data/pdf_sample2/").load_data()

In [5]:
# FAISS 인덱스 생성
dimension = 768 # 보통 768개
faiss_index = faiss.IndexFlatL2(dimension) # 유사도 분석을 이해 L2 사용

In [6]:
# FAISS를 LlamaIndex의 인덱싱 및 검색 파이프라인에 통합
vector_store = FaissVectorStore(faiss_index = faiss_index)
storage_context = StorageContext.from_defaults(vector_store = vector_store)

In [7]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(
    documents = documents,
    storage_context = storage_context,
    embed_model = embed_model,
    show_progress = True
)

/Users/restitutor/Documents/Restitutor/Workspace/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 12/12 [00:01<00:00,  7.69it/s]


In [8]:
# 저장할 디렉토리
persist_dir = "./faiss_storage"

# 인덱스 저장
index.storage_context.persist(persist_dir = persist_dir)

In [9]:
# 저장된 FAISS 벡터 스토어
loaded_vector_store = FaissVectorStore.from_persist_dir(persist_dir = persist_dir)

2026-05-19 11:31:26,859 - INFO - Loading llama_index.vector_stores.faiss.base from ./faiss_storage/default__vector_store.json.


In [13]:
# 저장된 인덱스 로드 위치 정의
loaded_storage_context = StorageContext.from_defaults(
    vector_store = loaded_vector_store,
    persist_dir = persist_dir
)

# LlamaIndex의 메타 데이터를 저장
loaded_index = load_index_from_storage(
    storage_context = loaded_storage_context,
    embed_model = embed_model
)

2026-05-19 11:31:37,409 - INFO - Loading all indices.


In [14]:
# 쿼리 엔진
loaded_query_engine = loaded_index.as_query_engine(llm = llm)

2026-05-19 11:31:41,003 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [17]:
# 쿼리 실행
query = "what is advantage of models in thie documents?"
response = loaded_query_engine.query(query)

# 응답 출력
print(f"질문: {query}")
print(f"답변: {response}")

2026-05-19 11:32:08,999 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:32:15,292 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


질문: what is advantage of models in thie documents?
답변: The models described in this document offer several advantages over traditional models.  Firstly, they achieve significantly faster training times compared to recurrent or convolutional layer-based architectures. Secondly, these models demonstrate superior performance on translation tasks, achieving state-of-the-art results on both the WMT 2014 English-to-German and English-to-French translation tasks. Thirdly, the use of attention mechanisms allows for a more efficient handling of long sequences compared to traditional recurrent or convolutional layers, enabling models to perform well with larger inputs like images or videos.  



In [18]:
# 쿼리 실행
query = "이 논문에서 제안하는 모델의 장점은 무엇인가요? 답변은 한국어로 해주세요"
response = loaded_query_engine.query(query)

# 응답 출력
print(f"질문: {query}")
print(f"답변: {response}")

2026-05-19 11:32:15,318 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:32:18,636 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


질문: 이 논문에서 제안하는 모델의 장점은 무엇인가요? 답변은 한국어로 해주세요
답변: 이 논문에서는 Transformer 모델을 제시하며,  residual connection과 layer normalization을 사용하여 효율적인 모델 구조를 제공합니다. 또한, encoder와 decoder의 각 단계에서 multi-head attention을 적용하여 정보 처리능력을 강화했습니다. 이러한 장점들은 모델의 빠른 성능 향상 및 높은 정확도 달성에 기여할 수 있습니다. 



### 파일에서 불러온 내용

In [19]:
# 쿼리 엔진
query_engine1 = loaded_index.as_query_engine(llm = llm)

In [20]:
# 쿼리 실행
query = "이 논문에서 제안하는 모델의 장점은 무엇인가요? 답변은 한국어로 해주세요"
response = query_engine1.query(query)

# 응답 출력
print(f"질문: {query}")
print(f"답변: {response}")

2026-05-19 11:32:45,133 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:32:46,672 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


질문: 이 논문에서 제안하는 모델의 장점은 무엇인가요? 답변은 한국어로 해주세요
답변: 이 논문에서 제안하는 모델의 장점은 **linear time**으로 가능한 **Neural Machine Translation**을 제공한다는 점입니다. 또한, **residual connection**, **layer normalization**과 같은 구조를 사용하여 model의 성능을 향상시켰습니다. 

